# Streaming Simulado — Indicador Criança Alfabetizada

Este notebook demonstra a ingestão de eventos educacionais em tempo quase real para o Tech Challenge — Fase 2.

## Objetivo

Simular registros chegando individualmente, em pequenos intervalos, como ocorreria em uma arquitetura orientada a eventos.

## Estratégia

Serão utilizados registros públicos reais como base para um replay controlado. O valor educacional será real, mas o horário de chegada será simulado.

Também serão incluídos alguns eventos inválidos propositalmente para demonstrar:

- validação de schema;
- aplicação de regras de negócio;
- separação entre eventos aceitos e rejeitados;
- medição de latência e volume;
- geração de logs.

## Isolamento dos dados

Os eventos simulados não serão gravados nos datasets oficiais `bronze`, `silver` ou `gold`.

Os resultados serão armazenados localmente em:

- JSONL;
- Parquet;
- arquivos de eventos aceitos;
- arquivos de eventos rejeitados;
- logs da execução.

## Fluxo

Registro público real → Evento simulado → Validação → Aceito ou rejeitado → JSONL/Parquet

> Em uma arquitetura de produção, esse fluxo poderia ser implementado com Pub/Sub, Cloud Run e BigQuery.


1. CONFIGURAÇÃO SEGURA DO STREAMING SIMULADO

In [1]:

from datetime import datetime, timezone
from pathlib import Path

import json
import random
import time
import uuid

import pandas as pd


# ------------------------------------------------------------
# Identificação da execução
# ------------------------------------------------------------

execution_id = str(uuid.uuid4())
started_at = datetime.now(timezone.utc)


# ------------------------------------------------------------
# Trava de segurança
# ------------------------------------------------------------

OFFICIAL_DATASETS = {
    "bronze",
    "silver",
    "gold",
}

TARGET_ENVIRONMENT = "local_simulation"

if TARGET_ENVIRONMENT in OFFICIAL_DATASETS:
    raise RuntimeError(
        "Execução bloqueada: eventos simulados não podem ser "
        "gravados em datasets oficiais."
    )


# ------------------------------------------------------------
# Estrutura local de armazenamento
# ------------------------------------------------------------

BASE_DIRECTORY = Path("/content/streaming_simulado")

ACCEPTED_DIRECTORY = BASE_DIRECTORY / "eventos_aceitos"
REJECTED_DIRECTORY = BASE_DIRECTORY / "eventos_rejeitados"
LOGS_DIRECTORY = BASE_DIRECTORY / "logs"
PARQUET_DIRECTORY = BASE_DIRECTORY / "parquet"

for directory in [
    ACCEPTED_DIRECTORY,
    REJECTED_DIRECTORY,
    LOGS_DIRECTORY,
    PARQUET_DIRECTORY,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Parâmetros da simulação
# ------------------------------------------------------------

NUMBER_OF_EVENTS = 20

MINIMUM_INTERVAL_SECONDS = 0.3
MAXIMUM_INTERVAL_SECONDS = 1.0

VALID_EVENT_TYPES = {
    "atualizacao_indicador",
    "nova_medicao_desempenho",
    "atualizacao_meta",
}


# ------------------------------------------------------------
# Resumo inicial
# ------------------------------------------------------------

print("=" * 70)
print("STREAMING SIMULADO — AMBIENTE PREPARADO")
print("=" * 70)

print(f"Execution ID: {execution_id}")
print(f"Início UTC: {started_at.isoformat()}")
print(f"Eventos planejados: {NUMBER_OF_EVENTS}")
print(f"Ambiente de destino: {TARGET_ENVIRONMENT}")
print(f"Diretório principal: {BASE_DIRECTORY}")

print("\nDatasets oficiais protegidos:")
for dataset in sorted(OFFICIAL_DATASETS):
    print(f"- {dataset}")

print("\nNenhum dado foi enviado ao BigQuery.")

STREAMING SIMULADO — AMBIENTE PREPARADO
Execution ID: 87d765dd-512f-4158-bf65-44e1402c597d
Início UTC: 2026-07-12T14:13:38.720831+00:00
Eventos planejados: 20
Ambiente de destino: local_simulation
Diretório principal: /content/streaming_simulado

Datasets oficiais protegidos:
- bronze
- gold
- silver

Nenhum dado foi enviado ao BigQuery.


## Fonte dos eventos simulados

Para evitar a criação de resultados educacionais fictícios, a simulação utilizará registros públicos reais já tratados na tabela `gold.indicador_municipio`.

Serão selecionados registros de 2024 com:

- município identificado;
- taxa de alfabetização disponível;
- percentual de participação disponível;
- meta correspondente ao ano disponível.

Esses registros serão transformados em eventos individuais.

O conteúdo educacional continuará sendo real. Serão simulados apenas:

- o identificador do evento;
- o instante de chegada;
- o intervalo entre os eventos;
- o comportamento de processamento em tempo quase real.

A consulta realizada nesta etapa é somente de leitura e não modifica os datasets oficiais.

2. AUTENTICAÇÃO E LEITURA DOS REGISTROS REAIS

In [3]:


from google.colab import auth
from google.cloud import bigquery


# ------------------------------------------------------------
# Autenticação no Google Cloud
# ------------------------------------------------------------

auth.authenticate_user()


# ------------------------------------------------------------
# Configuração somente para leitura
# ------------------------------------------------------------

PROJECT_ID = "macro-coil-475920-k5"
LOCATION = "US"

SOURCE_TABLE = (
    f"{PROJECT_ID}.gold.indicador_municipio"
)

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)


# ------------------------------------------------------------
# Consulta de registros reais elegíveis
# ------------------------------------------------------------

query = f"""
SELECT
  ano,
  id_municipio,
  nome_municipio,
  sigla_uf,
  regiao,
  rede,
  taxa_alfabetizacao_observada,
  percentual_participacao,
  meta_ano_resultado,
  diferenca_resultado_meta_pp,
  status_cumprimento_meta

FROM
  `{SOURCE_TABLE}`

WHERE
  ano = 2024
  AND id_municipio IS NOT NULL
  AND nome_municipio IS NOT NULL
  AND taxa_alfabetizacao_observada IS NOT NULL
  AND percentual_participacao IS NOT NULL
  AND meta_ano_resultado IS NOT NULL

ORDER BY
  FARM_FINGERPRINT(
    CONCAT(
      id_municipio,
      '|',
      rede
    )
  )

LIMIT 15
"""


real_records_df = (
    client
    .query(query, location=LOCATION)
    .result()
    .to_dataframe()
)


# ------------------------------------------------------------
# Quality gate da extração
# ------------------------------------------------------------

EXPECTED_REAL_RECORDS = 15

if len(real_records_df) != EXPECTED_REAL_RECORDS:
    raise RuntimeError(
        "A consulta não retornou os 15 registros reais esperados."
    )


required_columns = {
    "ano",
    "id_municipio",
    "nome_municipio",
    "sigla_uf",
    "regiao",
    "rede",
    "taxa_alfabetizacao_observada",
    "percentual_participacao",
    "meta_ano_resultado",
    "diferenca_resultado_meta_pp",
    "status_cumprimento_meta",
}

missing_columns = (
    required_columns
    - set(real_records_df.columns)
)

if missing_columns:
    raise RuntimeError(
        f"Colunas obrigatórias ausentes: {missing_columns}"
    )


print("=" * 70)
print("REGISTROS REAIS CARREGADOS PARA O REPLAY")
print("=" * 70)

print(f"Tabela consultada: {SOURCE_TABLE}")
print(f"Quantidade carregada: {len(real_records_df)}")
print("Modo de acesso: somente leitura")
print("Nenhuma tabela do BigQuery foi modificada.")

display(real_records_df)

REGISTROS REAIS CARREGADOS PARA O REPLAY
Tabela consultada: macro-coil-475920-k5.gold.indicador_municipio
Quantidade carregada: 15
Modo de acesso: somente leitura
Nenhuma tabela do BigQuery foi modificada.


,ano,id_municipio,nome_municipio,sigla_uf,regiao,rede,taxa_alfabetizacao_observada,percentual_participacao,meta_ano_resultado,diferenca_resultado_meta_pp,status_cumprimento_meta
0,2024,3533403,Nova Odessa,SP,Sudeste,Municipal,57.96,92.82,59.09,-1.13,meta_nao_atingida
1,2024,3527009,Lindóia,SP,Sudeste,Municipal,61.13,91.67,77.54,-16.41,meta_nao_atingida
2,2024,3137205,Lagoa da Prata,MG,Sudeste,Municipal,79.90,96.76,73.73,6.17,meta_atingida
3,2024,1718881,Santa Maria do Tocantins,TO,Norte,Municipal,39.73,83.67,31.78,7.95,meta_atingida
4,2024,3550001,São Luís do Paraitinga,SP,Sudeste,Municipal,59.66,94.62,74.44,-14.78,meta_nao_atingida
5,2024,3502101,Andradina,SP,Sudeste,Municipal,65.48,94.38,60.79,4.69,meta_atingida
6,2024,2925402,Potiraguá,BA,Nordeste,Municipal,27.54,92.00,35.06,-7.52,meta_nao_atingida
7,2024,2203602,Eliseu Martins,PI,Nordeste,Municipal,65.85,100.00,80.00,-14.15,meta_nao_atingida
8,2024,4300570,Alto Feliz,RS,Sul,Municipal,58.30,81.58,73.84,-15.54,meta_nao_atingida
9,2024,4107009,Curiúva,PR,Sul,Municipal,68.88,88.83,80.00,-11.12,meta_nao_atingida


## 3. Construção da fila de eventos

Nesta etapa, os registros reais serão transformados em mensagens individuais.

Serão criados:

- 15 eventos válidos;
- 5 eventos inválidos controlados;
- uma fila com 20 eventos no total.

In [7]:


valid_events = []


# Distribuição planejada dos 15 eventos reais
event_type_sequence = (
    ["atualizacao_indicador"] * 8
    + ["atualizacao_meta"] * 4
    + ["nova_medicao_desempenho"] * 3
)


# ------------------------------------------------------------
# Transformação dos registros reais em eventos válidos
# ------------------------------------------------------------

for index, (_, row) in enumerate(
    real_records_df.iterrows()
):

    event_type = event_type_sequence[index]

    event = {
        "event_id": str(uuid.uuid4()),
        "execution_id": execution_id,
        "tipo_evento": event_type,

        "ano_referencia": int(row["ano"]),
        "id_municipio": str(row["id_municipio"]),
        "nome_municipio": str(row["nome_municipio"]),
        "sigla_uf": str(row["sigla_uf"]),
        "regiao": str(row["regiao"]),
        "rede": str(row["rede"]),

        "origem_dado": SOURCE_TABLE,
        "evento_simulado": True,
    }

    # --------------------------------------------------------
    # Campos específicos de cada tipo de evento
    # --------------------------------------------------------

    if event_type == "atualizacao_indicador":

        event.update(
            {
                "taxa_alfabetizacao": float(
                    row["taxa_alfabetizacao_observada"]
                ),
                "percentual_participacao": float(
                    row["percentual_participacao"]
                ),
                "meta_alfabetizacao": float(
                    row["meta_ano_resultado"]
                ),
                "diferenca_resultado_meta_pp": float(
                    row["diferenca_resultado_meta_pp"]
                ),
                "status_cumprimento_meta": str(
                    row["status_cumprimento_meta"]
                ),
            }
        )

    elif event_type == "atualizacao_meta":

        event.update(
            {
                "meta_alfabetizacao": float(
                    row["meta_ano_resultado"]
                ),
            }
        )

    elif event_type == "nova_medicao_desempenho":

        event.update(
            {
                "taxa_alfabetizacao": float(
                    row["taxa_alfabetizacao_observada"]
                ),
                "percentual_participacao": float(
                    row["percentual_participacao"]
                ),
            }
        )

    valid_events.append(event)


# ------------------------------------------------------------
# Cinco eventos inválidos criados propositalmente
# ------------------------------------------------------------

invalid_events = [

    # 1. Tipo de evento desconhecido
    {
        "event_id": str(uuid.uuid4()),
        "execution_id": execution_id,
        "tipo_evento": "evento_desconhecido",
        "ano_referencia": 2024,
        "id_municipio": "3550308",
        "taxa_alfabetizacao": 65.0,
        "origem_dado": "evento_teste_controlado",
        "evento_simulado": True,
    },

    # 2. Município ausente
    {
        "event_id": str(uuid.uuid4()),
        "execution_id": execution_id,
        "tipo_evento": "atualizacao_indicador",
        "ano_referencia": 2024,
        "id_municipio": None,
        "taxa_alfabetizacao": 70.0,
        "percentual_participacao": 85.0,
        "meta_alfabetizacao": 68.0,
        "origem_dado": "evento_teste_controlado",
        "evento_simulado": True,
    },

    # 3. Taxa de alfabetização impossível
    {
        "event_id": str(uuid.uuid4()),
        "execution_id": execution_id,
        "tipo_evento": "nova_medicao_desempenho",
        "ano_referencia": 2024,
        "id_municipio": "3550308",
        "taxa_alfabetizacao": 130.0,
        "percentual_participacao": 90.0,
        "origem_dado": "evento_teste_controlado",
        "evento_simulado": True,
    },

    # 4. Ano de referência ausente
    {
        "event_id": str(uuid.uuid4()),
        "execution_id": execution_id,
        "tipo_evento": "atualizacao_indicador",
        "ano_referencia": None,
        "id_municipio": "3550308",
        "taxa_alfabetizacao": 65.0,
        "percentual_participacao": 82.0,
        "meta_alfabetizacao": 67.0,
        "origem_dado": "evento_teste_controlado",
        "evento_simulado": True,
    },

    # 5. Meta superior a 100%
    {
        "event_id": str(uuid.uuid4()),
        "execution_id": execution_id,
        "tipo_evento": "atualizacao_meta",
        "ano_referencia": 2024,
        "id_municipio": "3550308",
        "meta_alfabetizacao": 150.0,
        "origem_dado": "evento_teste_controlado",
        "evento_simulado": True,
    },
]


# ------------------------------------------------------------
# Montagem e embaralhamento da fila
# ------------------------------------------------------------

event_queue = valid_events + invalid_events

random.shuffle(event_queue)


# ------------------------------------------------------------
# Quality gate da fila
# ------------------------------------------------------------

if len(valid_events) != 15:
    raise RuntimeError(
        "Quantidade inesperada de eventos válidos."
    )

if len(invalid_events) != 5:
    raise RuntimeError(
        "Quantidade inesperada de eventos inválidos."
    )

if len(event_queue) != NUMBER_OF_EVENTS:
    raise RuntimeError(
        "A fila não contém os 20 eventos planejados."
    )


# ------------------------------------------------------------
# Resumo da fila
# ------------------------------------------------------------

queue_summary_df = pd.DataFrame(
    {
        "categoria": [
            "Eventos válidos",
            "Eventos inválidos controlados",
            "Total da fila",
        ],
        "quantidade": [
            len(valid_events),
            len(invalid_events),
            len(event_queue),
        ],
    }
)

print("=" * 70)
print("FILA DE EVENTOS CONSTRUÍDA")
print("=" * 70)

display(queue_summary_df)

print("\nDistribuição por tipo de evento:")

event_types_df = (
    pd.DataFrame(event_queue)
    ["tipo_evento"]
    .value_counts()
    .rename_axis("tipo_evento")
    .reset_index(name="quantidade")
)

display(event_types_df)

print("\nExemplo de evento da fila:")

print(
    json.dumps(
        event_queue[0],
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNenhum evento foi processado ou enviado ao BigQuery.")

FILA DE EVENTOS CONSTRUÍDA


,categoria,quantidade
0,Eventos válidos,15
1,Eventos inválidos controlados,5
2,Total da fila,20



Distribuição por tipo de evento:


,tipo_evento,quantidade
0,atualizacao_indicador,10
1,atualizacao_meta,5
2,nova_medicao_desempenho,4
3,evento_desconhecido,1



Exemplo de evento da fila:
{
  "event_id": "ed9b1e6a-3b17-4401-826e-edb384312cad",
  "execution_id": "87d765dd-512f-4158-bf65-44e1402c597d",
  "tipo_evento": "atualizacao_meta",
  "ano_referencia": 2024,
  "id_municipio": "3550308",
  "meta_alfabetizacao": 150.0,
  "origem_dado": "evento_teste_controlado",
  "evento_simulado": true
}

Nenhum evento foi processado ou enviado ao BigQuery.


## 4. Validação e processamento em tempo quase real

Nesta etapa, os 20 eventos serão processados individualmente.

Entre cada mensagem haverá um pequeno intervalo aleatório para simular a chegada contínua de eventos.

Para cada evento, o pipeline irá:

- registrar o horário de recebimento;
- validar os campos obrigatórios;
- verificar o tipo do evento;
- validar o código do município;
- conferir o ano de referência;
- verificar se taxas, metas e percentuais estão entre 0% e 100%;
- classificar o evento como aceito ou rejeitado;
- registrar os motivos de rejeição;
- medir a latência de processamento.

Os dados continuarão restritos ao ambiente local do Colab.

In [8]:
def valor_ausente(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def validar_percentual(event, field_name, errors):
    value = event.get(field_name)

    if valor_ausente(value):
        errors.append(f"Campo obrigatório ausente: {field_name}")
        return

    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        errors.append(f"Campo não numérico: {field_name}")
        return

    if numeric_value < 0 or numeric_value > 100:
        errors.append(
            f"{field_name} fora do intervalo de 0 a 100"
        )


def validar_evento(event):
    errors = []

    required_common_fields = [
        "event_id",
        "execution_id",
        "tipo_evento",
        "ano_referencia",
        "id_municipio",
        "origem_dado",
        "evento_simulado",
    ]

    for field_name in required_common_fields:
        if valor_ausente(event.get(field_name)):
            errors.append(
                f"Campo obrigatório ausente: {field_name}"
            )

    event_type = event.get("tipo_evento")

    if event_type not in VALID_EVENT_TYPES:
        errors.append(
            f"Tipo de evento inválido: {event_type}"
        )

    year = event.get("ano_referencia")

    if not valor_ausente(year):
        try:
            year = int(year)

            if year < 2023 or year > 2030:
                errors.append(
                    "Ano de referência fora do intervalo esperado"
                )
        except (TypeError, ValueError):
            errors.append(
                "Ano de referência deve ser inteiro"
            )

    municipality_id = event.get("id_municipio")

    if not valor_ausente(municipality_id):
        municipality_id = str(municipality_id).strip()

        if (
            not municipality_id.isdigit()
            or len(municipality_id) != 7
        ):
            errors.append(
                "ID do município deve possuir 7 dígitos"
            )

    if event.get("evento_simulado") is not True:
        errors.append(
            "Evento sem identificação explícita de simulação"
        )

    if event_type == "atualizacao_indicador":
        validar_percentual(
            event,
            "taxa_alfabetizacao",
            errors,
        )

        validar_percentual(
            event,
            "percentual_participacao",
            errors,
        )

        validar_percentual(
            event,
            "meta_alfabetizacao",
            errors,
        )

    elif event_type == "atualizacao_meta":
        validar_percentual(
            event,
            "meta_alfabetizacao",
            errors,
        )

    elif event_type == "nova_medicao_desempenho":
        validar_percentual(
            event,
            "taxa_alfabetizacao",
            errors,
        )

        validar_percentual(
            event,
            "percentual_participacao",
            errors,
        )

    return errors


accepted_events = []
rejected_events = []
processing_logs = []


print("=" * 70)
print("INÍCIO DO PROCESSAMENTO DOS EVENTOS")
print("=" * 70)


for sequence, event in enumerate(event_queue, start=1):
    interval_seconds = round(
        random.uniform(
            MINIMUM_INTERVAL_SECONDS,
            MAXIMUM_INTERVAL_SECONDS,
        ),
        2,
    )

    time.sleep(interval_seconds)

    received_at = datetime.now(timezone.utc)
    processing_timer = time.perf_counter()

    validation_errors = validar_evento(event)

    processed_at = datetime.now(timezone.utc)

    processing_latency_ms = round(
        (
            time.perf_counter()
            - processing_timer
        ) * 1000,
        3,
    )

    processed_event = event.copy()

    processed_event.update(
        {
            "sequencia_evento": sequence,
            "data_recebimento_utc": received_at.isoformat(),
            "data_processamento_utc": processed_at.isoformat(),
            "intervalo_antes_evento_segundos": interval_seconds,
            "latencia_processamento_ms": processing_latency_ms,
        }
    )

    if validation_errors:
        status = "REJEITADO"

        processed_event["status_processamento"] = status
        processed_event["motivos_rejeicao"] = validation_errors

        rejected_events.append(processed_event)

    else:
        status = "ACEITO"

        processed_event["status_processamento"] = status
        processed_event["motivos_rejeicao"] = []

        accepted_events.append(processed_event)

    processing_logs.append(
        {
            "execution_id": execution_id,
            "sequencia_evento": sequence,
            "event_id": event.get("event_id"),
            "tipo_evento": event.get("tipo_evento"),
            "id_municipio": event.get("id_municipio"),
            "status_processamento": status,
            "quantidade_erros": len(validation_errors),
            "motivos_rejeicao": " | ".join(validation_errors),
            "intervalo_antes_evento_segundos": interval_seconds,
            "latencia_processamento_ms": processing_latency_ms,
            "data_recebimento_utc": received_at.isoformat(),
            "data_processamento_utc": processed_at.isoformat(),
        }
    )

    print(
        f"[{sequence:02d}/{len(event_queue)}] "
        f"{status} | "
        f"{event.get('tipo_evento')} | "
        f"municipio={event.get('id_municipio')}"
    )


accepted_events_df = pd.DataFrame(accepted_events)
rejected_events_df = pd.DataFrame(rejected_events)
processing_logs_df = pd.DataFrame(processing_logs)


processing_summary_df = pd.DataFrame(
    {
        "resultado": [
            "Eventos recebidos",
            "Eventos aceitos",
            "Eventos rejeitados",
        ],
        "quantidade": [
            len(event_queue),
            len(accepted_events),
            len(rejected_events),
        ],
    }
)


print("\n" + "=" * 70)
print("RESUMO DO PROCESSAMENTO")
print("=" * 70)

display(processing_summary_df)

print("\nEventos rejeitados e motivos:")

display(
    processing_logs_df[
        processing_logs_df[
            "status_processamento"
        ] == "REJEITADO"
    ][
        [
            "sequencia_evento",
            "tipo_evento",
            "id_municipio",
            "motivos_rejeicao",
        ]
    ]
)

print("\nNenhum dado foi enviado ao BigQuery.")

INÍCIO DO PROCESSAMENTO DOS EVENTOS
[01/20] REJEITADO | atualizacao_meta | municipio=3550308
[02/20] ACEITO | atualizacao_indicador | municipio=1718881
[03/20] ACEITO | nova_medicao_desempenho | municipio=3100302
[04/20] ACEITO | atualizacao_meta | municipio=4300570
[05/20] ACEITO | atualizacao_indicador | municipio=3502101
[06/20] ACEITO | atualizacao_indicador | municipio=3550001
[07/20] ACEITO | nova_medicao_desempenho | municipio=2312502
[08/20] ACEITO | atualizacao_indicador | municipio=2203602
[09/20] ACEITO | atualizacao_indicador | municipio=3137205
[10/20] ACEITO | atualizacao_meta | municipio=4107009
[11/20] REJEITADO | nova_medicao_desempenho | municipio=3550308
[12/20] REJEITADO | evento_desconhecido | municipio=3550308
[13/20] ACEITO | atualizacao_meta | municipio=2513406
[14/20] ACEITO | nova_medicao_desempenho | municipio=4106308
[15/20] REJEITADO | atualizacao_indicador | municipio=None
[16/20] ACEITO | atualizacao_indicador | municipio=3527009
[17/20] REJEITADO | atual

,resultado,quantidade
0,Eventos recebidos,20
1,Eventos aceitos,15
2,Eventos rejeitados,5



Eventos rejeitados e motivos:


,sequencia_evento,tipo_evento,id_municipio,motivos_rejeicao
0,1,atualizacao_meta,3550308,meta_alfabetizacao fora do intervalo de 0 a 100
10,11,nova_medicao_desempenho,3550308,taxa_alfabetizacao fora do intervalo de 0 a 100
11,12,evento_desconhecido,3550308,Tipo de evento inválido: evento_desconhecido
14,15,atualizacao_indicador,None,Campo obrigatório ausente: id_municipio
16,17,atualizacao_indicador,3550308,Campo obrigatório ausente: ano_referencia



Nenhum dado foi enviado ao BigQuery.


## 5. Persistência dos eventos e quality gate final

Após o processamento, os resultados serão armazenados localmente em diferentes formatos.

### Arquivos gerados

- Eventos aceitos em JSONL;
- Eventos rejeitados em JSONL;
- Eventos aceitos em Parquet;
- Eventos rejeitados em Parquet;
- Logs detalhados em CSV e JSONL;
- Resumo operacional da execução;
- Manifesto com os arquivos produzidos.

O formato JSONL preserva cada evento como uma mensagem independente.

O formato Parquet demonstra uma alternativa colunar e compacta para armazenamento analítico.

O quality gate final verificará:

- recebimento dos 20 eventos planejados;
- aceite dos 15 eventos válidos;
- rejeição dos 5 eventos inválidos;
- identificação das cinco regras de erro;
- existência dos arquivos produzidos;
- manutenção do isolamento em relação aos datasets oficiais.

In [9]:
finished_at = datetime.now(timezone.utc)

safe_execution_id = execution_id.replace("-", "_")

accepted_jsonl_path = (
    ACCEPTED_DIRECTORY
    / f"eventos_aceitos_{safe_execution_id}.jsonl"
)

rejected_jsonl_path = (
    REJECTED_DIRECTORY
    / f"eventos_rejeitados_{safe_execution_id}.jsonl"
)

accepted_parquet_path = (
    PARQUET_DIRECTORY
    / f"eventos_aceitos_{safe_execution_id}.parquet"
)

rejected_parquet_path = (
    PARQUET_DIRECTORY
    / f"eventos_rejeitados_{safe_execution_id}.parquet"
)

logs_csv_path = (
    LOGS_DIRECTORY
    / f"processamento_{safe_execution_id}.csv"
)

logs_jsonl_path = (
    LOGS_DIRECTORY
    / f"processamento_{safe_execution_id}.jsonl"
)

summary_csv_path = (
    LOGS_DIRECTORY
    / f"resumo_{safe_execution_id}.csv"
)

manifest_path = (
    LOGS_DIRECTORY
    / f"manifesto_{safe_execution_id}.json"
)


def gravar_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    default=str,
                )
                + "\n"
            )


def preparar_parquet(dataframe):
    prepared_dataframe = dataframe.copy()

    for column_name in prepared_dataframe.columns:
        if prepared_dataframe[column_name].dtype == "object":
            prepared_dataframe[column_name] = (
                prepared_dataframe[column_name].apply(
                    lambda value: (
                        json.dumps(
                            value,
                            ensure_ascii=False,
                            default=str,
                        )
                        if isinstance(value, (list, dict))
                        else (
                            None
                            if valor_ausente(value)
                            else str(value)
                        )
                    )
                )
            )

    return prepared_dataframe


gravar_jsonl(
    accepted_events,
    accepted_jsonl_path,
)

gravar_jsonl(
    rejected_events,
    rejected_jsonl_path,
)

accepted_parquet_df = preparar_parquet(
    accepted_events_df
)

rejected_parquet_df = preparar_parquet(
    rejected_events_df
)

accepted_parquet_df.to_parquet(
    accepted_parquet_path,
    index=False,
)

rejected_parquet_df.to_parquet(
    rejected_parquet_path,
    index=False,
)

processing_logs_df.to_csv(
    logs_csv_path,
    index=False,
)

processing_logs_df.to_json(
    logs_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

received_times = pd.to_datetime(
    processing_logs_df["data_recebimento_utc"],
    utc=True,
)

processed_times = pd.to_datetime(
    processing_logs_df["data_processamento_utc"],
    utc=True,
)

simulation_duration_seconds = round(
    (
        processed_times.max()
        - received_times.min()
    ).total_seconds(),
    2,
)

average_latency_ms = round(
    processing_logs_df[
        "latencia_processamento_ms"
    ].mean(),
    3,
)

p95_latency_ms = round(
    processing_logs_df[
        "latencia_processamento_ms"
    ].quantile(0.95),
    3,
)

throughput_events_per_second = round(
    len(event_queue)
    / simulation_duration_seconds,
    2,
) if simulation_duration_seconds > 0 else None

all_rejection_reasons = " | ".join(
    processing_logs_df.loc[
        processing_logs_df[
            "status_processamento"
        ] == "REJEITADO",
        "motivos_rejeicao",
    ].tolist()
)

expected_error_fragments = [
    "meta_alfabetizacao fora do intervalo",
    "taxa_alfabetizacao fora do intervalo",
    "Tipo de evento inválido",
    "Campo obrigatório ausente: id_municipio",
    "Campo obrigatório ausente: ano_referencia",
]

expected_errors_detected = all(
    fragment in all_rejection_reasons
    for fragment in expected_error_fragments
)

generated_files = [
    accepted_jsonl_path,
    rejected_jsonl_path,
    accepted_parquet_path,
    rejected_parquet_path,
    logs_csv_path,
    logs_jsonl_path,
]

all_files_created = all(
    path.exists()
    for path in generated_files
)

official_datasets_protected = (
    TARGET_ENVIRONMENT
    not in OFFICIAL_DATASETS
)

quality_gate_passed = (
    len(event_queue) == 20
    and len(accepted_events) == 15
    and len(rejected_events) == 5
    and expected_errors_detected
    and all_files_created
    and official_datasets_protected
)

summary = {
    "execution_id": execution_id,
    "inicio_utc": started_at.isoformat(),
    "fim_utc": finished_at.isoformat(),
    "eventos_recebidos": len(event_queue),
    "eventos_aceitos": len(accepted_events),
    "eventos_rejeitados": len(rejected_events),
    "percentual_aceitos": round(
        len(accepted_events)
        / len(event_queue)
        * 100,
        2,
    ),
    "percentual_rejeitados": round(
        len(rejected_events)
        / len(event_queue)
        * 100,
        2,
    ),
    "duracao_simulacao_segundos": (
        simulation_duration_seconds
    ),
    "latencia_media_ms": average_latency_ms,
    "latencia_p95_ms": p95_latency_ms,
    "eventos_por_segundo": (
        throughput_events_per_second
    ),
    "erros_controlados_detectados": (
        expected_errors_detected
    ),
    "arquivos_criados": all_files_created,
    "datasets_oficiais_protegidos": (
        official_datasets_protected
    ),
    "status_pipeline": (
        "SUCESSO"
        if quality_gate_passed
        else "FALHA"
    ),
}

summary_df = pd.DataFrame([summary])

summary_df.to_csv(
    summary_csv_path,
    index=False,
)

manifest = {
    "execution_id": execution_id,
    "arquivos": [
        str(path)
        for path in generated_files
    ],
    "arquivo_resumo": str(summary_csv_path),
    "ambiente": TARGET_ENVIRONMENT,
    "evento_simulado": True,
}

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )

display(summary_df)

if not quality_gate_passed:
    raise RuntimeError(
        "Quality gate do streaming simulado reprovado."
    )

print("\nQUALITY GATE APROVADO")
print("Streaming simulado concluído com sucesso.")

print("\nArquivos gerados:")

for path in generated_files:
    print(f"- {path}")

print(f"- {summary_csv_path}")
print(f"- {manifest_path}")

print("\nNenhum dado foi enviado ao BigQuery.")

,execution_id,inicio_utc,fim_utc,eventos_recebidos,eventos_aceitos,eventos_rejeitados,percentual_aceitos,percentual_rejeitados,duracao_simulacao_segundos,latencia_media_ms,latencia_p95_ms,eventos_por_segundo,erros_controlados_detectados,arquivos_criados,datasets_oficiais_protegidos,status_pipeline
0,87d765dd-512f-4158-bf65-44e1402c597d,2026-07-12T14:13:38.720831+00:00,2026-07-12T14:40:01.894588+00:00,20,15,5,75.0,25.0,14.24,0.05,0.072,1.4,True,True,True,SUCESSO



QUALITY GATE APROVADO
Streaming simulado concluído com sucesso.

Arquivos gerados:
- /content/streaming_simulado/eventos_aceitos/eventos_aceitos_87d765dd_512f_4158_bf65_44e1402c597d.jsonl
- /content/streaming_simulado/eventos_rejeitados/eventos_rejeitados_87d765dd_512f_4158_bf65_44e1402c597d.jsonl
- /content/streaming_simulado/parquet/eventos_aceitos_87d765dd_512f_4158_bf65_44e1402c597d.parquet
- /content/streaming_simulado/parquet/eventos_rejeitados_87d765dd_512f_4158_bf65_44e1402c597d.parquet
- /content/streaming_simulado/logs/processamento_87d765dd_512f_4158_bf65_44e1402c597d.csv
- /content/streaming_simulado/logs/processamento_87d765dd_512f_4158_bf65_44e1402c597d.jsonl
- /content/streaming_simulado/logs/resumo_87d765dd_512f_4158_bf65_44e1402c597d.csv
- /content/streaming_simulado/logs/manifesto_87d765dd_512f_4158_bf65_44e1402c597d.json

Nenhum dado foi enviado ao BigQuery.


## 6. Preservação dos artefatos no Google Drive

Os arquivos produzidos durante a simulação estão inicialmente no armazenamento temporário do Google Colab.

Nesta etapa, toda a pasta da execução será copiada para o Google Drive, garantindo a preservação de:

- eventos aceitos;
- eventos rejeitados;
- arquivos Parquet;
- logs CSV e JSONL;
- resumo operacional;
- manifesto da execução.

Esses artefatos poderão posteriormente ser organizados e versionados no repositório do projeto.

In [11]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

drive_destination = Path(
    "/content/drive/MyDrive/tech_challenge_alfabetizacao/"
    "streaming_simulado"
)

drive_destination.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if drive_destination.exists():
    shutil.rmtree(drive_destination)

shutil.copytree(
    BASE_DIRECTORY,
    drive_destination,
)

copied_files = [
    path
    for path in drive_destination.rglob("*")
    if path.is_file()
]

print("Artefatos preservados no Google Drive.")
print(f"Destino: {drive_destination}")
print(f"Quantidade de arquivos copiados: {len(copied_files)}")

for path in copied_files:
    print(f"- {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artefatos preservados no Google Drive.
Destino: /content/drive/MyDrive/tech_challenge_alfabetizacao/streaming_simulado
Quantidade de arquivos copiados: 8
- /content/drive/MyDrive/tech_challenge_alfabetizacao/streaming_simulado/eventos_aceitos/eventos_aceitos_87d765dd_512f_4158_bf65_44e1402c597d.jsonl
- /content/drive/MyDrive/tech_challenge_alfabetizacao/streaming_simulado/parquet/eventos_aceitos_87d765dd_512f_4158_bf65_44e1402c597d.parquet
- /content/drive/MyDrive/tech_challenge_alfabetizacao/streaming_simulado/parquet/eventos_rejeitados_87d765dd_512f_4158_bf65_44e1402c597d.parquet
- /content/drive/MyDrive/tech_challenge_alfabetizacao/streaming_simulado/eventos_rejeitados/eventos_rejeitados_87d765dd_512f_4158_bf65_44e1402c597d.jsonl
- /content/drive/MyDrive/tech_challenge_alfabetizacao/streaming_simulado/logs/resumo_87d765dd_512f_4158_bf65_44e1402c597d.csv
- 

## 7. Integração conceitual com a arquitetura Medalhão

Os arquivos Parquet de eventos aceitos representam a landing zone da ingestão streaming simulada.

Nesta demonstração, os eventos foram mantidos fora dos datasets oficiais para evitar a mistura entre dados públicos e registros simulados.

A tabela Gold foi utilizada somente como fonte de amostras reais para construir payloads verossímeis durante o replay. Ela não representa a origem dos eventos em uma arquitetura de produção.

Em produção, o fluxo seria:

Sistema produtor → Pub/Sub → Processamento → Bronze de eventos → Silver → Gold

Na simulação implementada, o fluxo equivalente é:

Replay em Python → Validação → Eventos aceitos em Parquet → Landing zone simulada

Portanto, a pipeline híbrida possui dois caminhos de entrada:

- Batch: dados históricos completos carregados no BigQuery;
- Streaming: atualizações pontuais processadas evento a evento e persistidas em uma landing zone simulada.

A consolidação física dos eventos na Bronze dependeria de serviços de streaming habilitados em uma conta GCP com faturamento.